# Paso 1:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random
import os

from torchvision import transforms
from sklearn.preprocessing import normalize
from PIL import Image
from tqdm import tqdm
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


# Paso 2

In [2]:
# Descargar el archivo zip desde Google Drive
!gdown --id 11-TD6add7zZaukIB8l2cVnBTJgsTjo8n

# Descomprimir el archivo
!unzip dataset_ecom_mini.zip

# Cargar los metadatos del conjunto de datos
data_dir = Path('eval')
df = pd.read_csv('eval.csv', delimiter=';')

print(f"El conjunto de datos contiene {len(df)} imágenes")
print(df.head())

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=11-TD6add7zZaukIB8l2cVnBTJgsTjo8n
To: /content/dataset_ecom_mini.zip
100% 10.6M/10.6M [00:00<00:00, 42.5MB/s]
Archive:  dataset_ecom_mini.zip
   creating: eval/
  inflating: eval/45elec.jpg         
  inflating: eval/im.qf53qh.input.jpg  
  inflating: eval/8toys.jpg          
  inflating: eval/im.rf7hhh.input.jpg  
  inflating: eval/im.rys7rv.input.jpg  
  inflating: eval/226pets.jpg        
  inflating: eval/0off.jpg           
  inflating: eval/im.7wd4b3.input.jpg  
  inflating: eval/443pets.jpg        
  inflating: eval/im.yy3v67.input.jpg  
  inflating: eval/im.zu5wxi.input.jpg  
  inflating: eval/39pets.jpg         
  inflating: eval/24elec.jpg         
  inflating: eval/131home.jpg        
  inflating: eval/im.w

# Paso 3:

In [5]:
def transform(image):
    transform_pipeline = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform_pipeline(image)

# Codificar todas las imágenes
embeddings = []
valid_filenames = []

# Get all image filenames from the dataframe
image_filenames = df['Title'].values
data_dir = Path('eval') # Define data_dir here as well

for filename in tqdm(image_filenames):
    img_path = data_dir / f"{filename}.jpg"
    if img_path.exists():
        try:
            image = Image.open(img_path)
            transformed_image = transform(image)
            embeddings.append(transformed_image)
        except Exception as e:
            print(f"Error codificando {filename}: {e}")

# Convertir a array de numpy
print(f"\nCodificadas exitosamente {len(embeddings)} imágenes")

100%|██████████| 300/300 [00:01<00:00, 260.42it/s]


Codificadas exitosamente 300 imágenes


# Paso 4

In [6]:
import torch
import torch.hub
from pathlib import Path


dinov2_models = {
        'dinov2_vits14': 'facebookresearch/dinov2',
        'dinov2_vitb14': 'facebookresearch/dinov2',
        'dinov2_vitl14': 'facebookresearch/dinov2',
        'dinov2_vitg14': 'facebookresearch/dinov2',
}

dinov3_vitl16 = {
        'repo':'./dinov3-main',
        'model':'dinov3_vitl16',
        'source':'local',
        'weights':'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
}

dinov3_vit16plus = {
        'repo':'./dinov3-main',
        'model':'dinov3_vits16plus',
        'source':'local',
        'weights':'dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth'}

dinov3 = [dinov3_vitl16, dinov3_vit16plus]

# DINOv3 ViT models pretrained on web images
!gdown --id 1fH2rq53x6JY6zE_WBKH5vP_ZGq4jvj46
!gdown --id 1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m

# Download and unzip the dinov3-main repository
!gdown --id 1-LqJ2bS_T8XOx5TClDq31MCkPPoXO4bZ
!unzip dinov3-main.zip


# Función para cargar modelos DINO locales
def load_local_dino_models(local_dino_models):
    for model_info in local_dino_models:
        model = torch.hub.load(repo_or_dir=model_info['repo'], model=model_info['model'], source=model_info['source'], weights=model_info['weights'])
        model = model.to(device)
        model.eval()
        model_info['model'] = model
    return local_dino_models

def load_dino_models(dino_models: dict, device):

    # Cargar DINOv2
    for model_name, repo in dino_models.items():
        try:
            model = torch.hub.load(repo, model_name)
            model = model.to(device)
            model.eval()
            dino_models[model_name] = model
            print(f"{model_name} cargado")
        except Exception as e:
            print(f"Error con {model_name}: {e}")
    return dino_models

# Usar la función
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dino_models = load_dino_models(dinov2_models, device)
dinov3_models = load_local_dino_models(dinov3)

print(f"Modelos listos: {list(dino_models.keys()) + [model['model'] for model in dinov3_models]}")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1fH2rq53x6JY6zE_WBKH5vP_ZGq4jvj46
From (redirected): https://drive.google.com/uc?id=1fH2rq53x6JY6zE_WBKH5vP_ZGq4jvj46&confirm=t&uuid=f3b527f3-dac8-4320-b15b-9ae1ec0d34e0
To: /content/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth
100% 115M/115M [00:00<00:00, 116MB/s] 
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m
From (redirected): https://drive.google.com/uc?id=1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m&confirm=t&uuid=9f6fa2e5-4824-4c77-9b33-30af

/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 202MB/s]


dinov2_vits14 cargado


...

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitg14/dinov2_vitg14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitg14_pretrain.pth


100%|██████████| 4.23G/4.23G [01:43<00:00, 43.9MB/s]


dinov2_vitg14 cargado


FileNotFoundError: [Errno 2] No such file or directory: '/content/dinov3-main/hubconf.py'

In [ ]:
def load_dino_models():
    """Carga todos los modelos disponibles de DINOv2 y DINOv3"""



for model_name, repo, model_id in model_configs:
        try:
            model = torch.hub.load(repo, model_id)
            model.eval()
            dino_models[model_name] = model
            print(f"{model_name} cargado exitosamente")
        except Exception as e:
            print(f"No se pudo cargar {model_name}: {e}")

    return dino_models




    # Cargar el modelo DinoV3 desde torch hub
# https://github.com/facebookresearch/dinov3

!gdown --id 1-LqJ2bS_T8XOx5TClDq31MCkPPoXO4bZ
!gdown --id 1fH2rq53x6JY6zE_WBKH5vP_ZGq4jvj46
!unzip dinov3-main.zip
model_v3.hub.load(REPO_DIR, 'dinov3_vits16plus', source='local', weights=Path('dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth'))
dino_

# Cargar todos los modelos
dino_models = load_dino_models()

# !gdown --id 1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m
# dinov3_vitb16 = torch.hub.load(REPO_DIR, 'dinov3_vitb16', source='local', weights=Path('dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'))

# !gdown --id 1CfyML_a7PkVpt4Lhy-3yNxSfUgsFfgCs
dinov3_vitl16 = torch.hub.load(repo_or_dir='./dinov3-main',model='dinov3_vitl16', source='local', weights='dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')
dinov3_vitl16 = dinov3_vitl16.to(device)
dinov3_vitl16.eval()





print(f"Modelos cargados exitosamente: {len(dino_models)}")

model = model.to(device)
model.eval()

print(f"Dimensión de salida del modelo: {model.embed_dim}")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /home/mbustamc/.cache/torch/hub/main.zip


/home/mbustamc/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/mbustamc/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/mbustamc/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /home/mbustamc/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:01<00:00, 48.3MB/s]


✅ dinov2_vits14 cargado exitosamente


...

100%|██████████| 1.13G/1.13G [00:29<00:00, 41.3MB/s]


✅ dinov2_vitl14 cargado exitosamente


Using cache found in /home/mbustamc/.cache/torch/hub/facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitg14/dinov2_vitg14_pretrain.pth" to /home/mbustamc/.cache/torch/hub/checkpoints/dinov2_vitg14_pretrain.pth


 99%|█████████▉| 4.19G/4.23G [03:21<00:00, 55.0MB/s] 

# import torch.hub

def load_dino_models():
    """Carga todos los modelos disponibles de DINOv2 y DINOv3"""
    
    dino_models = {}
    model_configs = [
        # DINOv2
        ('dinov2_vits14', 'facebookresearch/dinov2', 'dinov2_vits14'),
        ('dinov2_vitb14', 'facebookresearch/dinov2', 'dinov2_vitb14'),
        ('dinov2_vitm14', 'facebookresearch/dinov2', 'dinov2_vitm14'),
        ('dinov2_vitl14', 'facebookresearch/dinov2', 'dinov2_vitl14'),
        ('dinov2_vitg14', 'facebookresearch/dinov2', 'dinov2_vitg14'),
        
        # DINOv2 Registered
        ('dinov2_vits14_reg', 'facebookresearch/dinov2', 'dinov2_vits14_reg'),
        ('dinov2_vitb14_reg', 'facebookresearch/dinov2', 'dinov2_vitb14_reg'),
        ('dinov2_vitm14_reg', 'facebookresearch/dinov2', 'dinov2_vitm14_reg'),
        ('dinov2_vitl14_reg', 'facebookresearch/dinov2', 'dinov2_vitl14_reg'),
        ('dinov2_vitg14_reg', 'facebookresearch/dinov2', 'dinov2_vitg14_reg'),
        
        # DINOv3 (intentar cargar)
        ('dinov3_vitb14', 'facebookresearch/dinov2', 'dinov3_vitb14'),
        ('dinov3_vitl14', 'facebookresearch/dinov2', 'dinov3_vitl14'),
        ('dinov3_vitg14', 'facebookresearch/dinov2', 'dinov3_vitg14'),
    ]
    
    for model_name, repo, model_id in model_configs:
        try:
            model = torch.hub.load(repo, model_id)
            model.eval()
            dino_models[model_name] = model
            print(f"✅ {model_name} cargado exitosamente")
        except Exception as e:
            print(f"❌ No se pudo cargar {model_name}: {e}")
    
    return dino_models

# Cargar todos los modelos
dino_models = load_dino_models()

print(f"\n🎯 Modelos cargados exitosamente: {len(dino_models)}")

model = model.to(device)
model.eval()

print("¡Modelo DinoV2 cargado exitosamente!")
print(f"Dimensión de salida del modelo: {model.embed_dim}")

# import torch.hub



In [ ]:
# Paso 5: Codificar imágenes usando el modelo preentrenad

In [ ]:
def encode_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(img_tensor)

    return features.cpu().numpy().flatten()


# Cargar el modelo preentrenado (por ejemplo, ResNet50)

# Obtener todos los nombres de archivos de imágenes del dataframe
image_filenames = df['Title'].values
n_images = len(image_filenames)

print(f"Codificando {n_images} imágenes...")

# Codificar todas las imágenes
embeddings = []
valid_filenames = []

for filename in tqdm(image_filenames):
    img_path = data_dir / f"{filename}.jpg"

    if img_path.exists():
        try:
            embedding = encode_image(img_path)
            embeddings.append(embedding)
            valid_filenames.append(filename)
        except Exception as e:
            print(f"Error codificando {filename}: {e}")

# Convertir a array de numpy
embeddings = np.array(embeddings)
print(f"\nCodificadas exitosamente {len(embeddings)} imágenes")
print(f"Forma del embedding: {embeddings.shape}")